# Round 3 notebook 01 — product reverse engineering

        This notebook classifies each product by its observed market microstructure rather than by labels alone:
        liquidity layers, wall coverage, spread regimes, trade side, and slow-EMA payoff distance.

In [1]:
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

from pathlib import Path
import math
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

DATA = Path("data/ROUND3")
DAYS = [0, 1, 2]

def load_round3():
    prices = []
    trades = []
    for day in DAYS:
        p = pd.read_csv(DATA / f"prices_round_3_day_{day}.csv", sep=";")
        t = pd.read_csv(DATA / f"trades_round_3_day_{day}.csv", sep=";")
        p["day"] = day
        t["day"] = day
        prices.append(p)
        trades.append(t)
    prices = pd.concat(prices, ignore_index=True)
    trades = pd.concat(trades, ignore_index=True)
    for col in prices.columns:
        if col.startswith(("bid_", "ask_")) or col == "mid_price":
            prices[col] = pd.to_numeric(prices[col], errors="coerce")
    trades["price"] = pd.to_numeric(trades["price"], errors="coerce")
    trades["quantity"] = pd.to_numeric(trades["quantity"], errors="coerce")
    return prices, trades

def add_book_features(prices):
    p = prices.copy()
    p["best_bid"] = p["bid_price_1"]
    p["best_ask"] = p["ask_price_1"]
    p["spread"] = p["best_ask"] - p["best_bid"]
    bid_cols = ["bid_price_1", "bid_price_2", "bid_price_3"]
    ask_cols = ["ask_price_1", "ask_price_2", "ask_price_3"]
    p["wall_bid"] = p[bid_cols].min(axis=1)
    p["wall_ask"] = p[ask_cols].max(axis=1)
    p["wall_mid"] = (p["wall_bid"] + p["wall_ask"]) / 2
    p["wall_spread"] = p["wall_ask"] - p["wall_bid"]
    p["n_bid_levels"] = p[bid_cols].notna().sum(axis=1)
    p["n_ask_levels"] = p[ask_cols].notna().sum(axis=1)
    p["mid_minus_wall"] = p["mid_price"] - p["wall_mid"]
    return p

def add_future_mid(prices, horizons=(1, 5, 10, 25, 50, 100, 200, 500, 1000, 2000, 5000)):
    p = prices.sort_values(["product", "day", "timestamp"]).copy()
    for h in horizons:
        p[f"fut_mid_{h}"] = p.groupby(["product", "day"])["mid_price"].shift(-h)
        p[f"ret_{h}"] = p[f"fut_mid_{h}"] - p["mid_price"]
    return p

def attach_trades(prices, trades):
    t = trades.merge(
        prices,
        left_on=["day", "timestamp", "symbol"],
        right_on=["day", "timestamp", "product"],
        how="left",
    )
    t["side"] = np.where(
        t["price"] == t["ask_price_1"],
        "buy_agg",
        np.where(t["price"] == t["bid_price_1"], "sell_agg", "other"),
    )
    return t

def maker_markout_table(trades_with_book, horizons=(1, 10, 50, 100, 200, 500, 1000, 2000, 5000)):
    t = trades_with_book.copy()
    for h in horizons:
        # PnL for a passive maker who sold to an aggressive buyer, or bought from an aggressive seller.
        t[f"mk_{h}"] = np.where(
            t["side"] == "buy_agg",
            t["price"] - t[f"fut_mid_{h}"],
            np.where(t["side"] == "sell_agg", t[f"fut_mid_{h}"] - t["price"], np.nan),
        )
    agg = {
        "n": ("price", "size"),
        "avg_qty": ("quantity", "mean"),
        "avg_price": ("price", "mean"),
    }
    for h in horizons:
        agg[f"mk_{h}"] = (f"mk_{h}", "mean")
    return t.groupby(["symbol", "side"]).agg(**agg).round(3).reset_index()

def slow_ema_payoff(prices, span=1000, horizons=(10, 25, 50, 100, 200, 500, 1000, 2000)):
    rows = []
    p = prices.sort_values(["product", "day", "timestamp"]).copy()
    alpha = 2 / (span + 1)
    p["ema"] = p.groupby(["product", "day"])["mid_price"].transform(
        lambda s: s.ewm(alpha=alpha, adjust=False).mean()
    )
    p["dev"] = p["mid_price"] - p["ema"]
    for product, g in p.groupby("product"):
        if g["mid_price"].std() == 0:
            rows.append({"product": product, "dev_q10": 0, "dev_q90": 0, "best_H": None, "edge_ticks": 0})
            continue
        lo, hi = g["dev"].quantile(0.10), g["dev"].quantile(0.90)
        low = g[g["dev"] <= lo]
        high = g[g["dev"] >= hi]
        best = None
        for h in horizons:
            low_ret = low.groupby("day")["mid_price"].shift(-h) if False else low[f"ret_{h}"]
            high_ret = high[f"ret_{h}"]
            edge = (low_ret.mean() - high_ret.mean()) / 2
            rec = (h, edge, low_ret.mean(), high_ret.mean())
            if best is None or edge > best[1]:
                best = rec
        rows.append({
            "product": product,
            "dev_q10": lo,
            "dev_q90": hi,
            "best_H": best[0],
            "edge_ticks": best[1],
            "low_future_move": best[2],
            "high_future_move": best[3],
        })
    return pd.DataFrame(rows).sort_values("edge_ticks", ascending=False).round(3)

In [2]:
prices_raw, trades_raw = load_round3()
prices = add_future_mid(add_book_features(prices_raw))
trades = attach_trades(prices, trades_raw)

display(Markdown("## Product-level structure"))
prod = prices.groupby("product").agg(
    rows=("timestamp", "size"),
    mid_mean=("mid_price", "mean"),
    mid_std=("mid_price", "std"),
    spread_med=("spread", "median"),
    wall_spread_med=("wall_spread", "median"),
    l2_bid_presence=("bid_price_2", lambda s: s.notna().mean()),
    l2_ask_presence=("ask_price_2", lambda s: s.notna().mean()),
    l3_presence=("bid_price_3", lambda s: s.notna().mean()),
).round(3)
display(prod)

display(Markdown("## Slow-anchor mean reversion payoff distance"))
payoff = slow_ema_payoff(prices, span=1000)
display(payoff)

display(Markdown("## Passive-maker markout by product and trade side"))
mm = maker_markout_table(trades)
display(mm[mm["n"] >= 5].sort_values(["symbol", "side"]))

## Product-level structure

,rows,mid_mean,mid_std,spread_med,wall_spread_med,l2_bid_presence,l2_ask_presence,l3_presence
product,,,,,,,,
HYDROGEL_PACK,30000,9990.807,31.935,16.0,21.0,1.000,1.000,0.016
VELVETFRUIT_EXTRACT,30000,5250.098,15.630,5.0,6.0,0.544,0.545,0.020
VEV_4000,30000,1250.110,15.647,21.0,26.0,1.000,1.000,0.009
VEV_4500,30000,750.110,15.640,16.0,20.0,1.000,1.000,0.009
VEV_5000,30000,255.022,14.376,6.0,7.0,0.646,0.637,0.006
VEV_5100,30000,166.805,12.743,4.0,5.0,0.428,0.422,0.004
VEV_5200,30000,95.549,9.664,3.0,3.0,0.243,0.246,0.002
VEV_5300,30000,46.760,6.228,2.0,2.0,0.146,0.148,0.000
VEV_5400,30000,15.952,3.429,1.0,1.0,0.048,0.050,0.000


## Slow-anchor mean reversion payoff distance

,product,dev_q10,dev_q90,best_H,edge_ticks,low_future_move,high_future_move
0,HYDROGEL_PACK,-25.577,26.477,2000.0,32.039,35.430,-28.648
2,VEV_4000,-13.900,14.648,1000.0,20.881,23.099,-18.664
3,VEV_4500,-13.846,14.633,1000.0,20.879,23.058,-18.700
1,VELVETFRUIT_EXTRACT,-13.886,14.654,1000.0,20.770,22.864,-18.676
4,VEV_5000,-12.984,13.598,1000.0,19.531,21.280,-17.782
5,VEV_5100,-11.534,11.997,1000.0,16.831,17.917,-15.745
6,VEV_5200,-8.717,8.890,1000.0,12.638,12.839,-12.436
7,VEV_5300,-5.499,5.599,1000.0,8.101,7.792,-8.410
8,VEV_5400,-2.684,2.536,1000.0,3.497,2.607,-4.387
9,VEV_5500,-1.213,1.146,1000.0,1.595,1.174,-2.017


## Passive-maker markout by product and trade side

,symbol,side,n,avg_qty,avg_price,mk_1,mk_10,mk_50,mk_100,mk_200,mk_500,mk_1000,mk_2000,mk_5000
0,HYDROGEL_PACK,buy_agg,524,4.042,9997.082,7.903,7.832,8.006,7.274,6.372,4.427,2.909,-1.186,-33.090
1,HYDROGEL_PACK,sell_agg,486,4.033,9983.374,7.941,7.891,8.646,8.282,8.977,8.825,12.336,16.599,47.081
2,VELVETFRUIT_EXTRACT,buy_agg,777,6.306,5252.393,1.706,1.736,1.512,1.353,1.112,0.977,0.082,-0.576,3.371
3,VELVETFRUIT_EXTRACT,other,6,6.333,5256.333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,VELVETFRUIT_EXTRACT,sell_agg,589,5.655,5247.890,2.349,2.308,2.511,2.254,2.055,2.163,4.222,3.710,-0.707
5,VEV_4000,buy_agg,226,2.013,1259.695,10.409,10.089,10.400,10.207,10.868,10.384,9.814,7.702,11.068
6,VEV_4000,sell_agg,238,2.038,1240.433,10.515,10.305,10.141,9.722,9.919,10.581,10.482,10.594,8.016
11,VEV_5200,sell_agg,17,3.647,85.176,1.206,1.000,1.618,1.000,1.688,3.906,8.800,9.357,20.857
14,VEV_5300,sell_agg,119,3.479,44.235,0.765,0.706,0.643,0.829,0.978,0.942,2.722,3.202,-0.560
15,VEV_5400,sell_agg,225,3.498,14.884,0.573,0.547,0.593,0.489,0.448,0.616,0.745,0.453,-1.555


## Reverse-engineered product map

        - `HYDROGEL_PACK`: independent wall/MM product with two persistent quote layers and strong long-horizon mean reversion.
        - `VELVETFRUIT_EXTRACT`: underlying, tighter spread, thinner walls, buy-aggressive external flow.
        - `VEV_4000` and `VEV_4500`: delta-1 vouchers; they behave like the underlying minus strike with large voucher spread.
        - `VEV_5000` to `VEV_5300`: near-money option surface, but historically sparse visible trade prints. Order-depth fills matter more than trade CSV counts.
        - `VEV_5400` and `VEV_5500`: tight-spread OTM vouchers dominated by sell-aggressive bot flow into the bid.
        - `VEV_6000` and `VEV_6500`: pinned at `0/1`; visible mid markout is not monetizable in `none/worse` unless the book crosses, which it does not.